# Intermediate Representation Deep Dive

This notebook provides a deep dive into the neuro-analog Intermediate Representation (IR) system, graph construction, and hardware annotation for energy modeling.

**Goal:** Understand how models are decomposed into circuit-level primitives for accurate energy estimation and hardware mapping.

**This is the bridge between hardware and neural architects:** it decomposes neural networks into circuit-level primitives that hardware architects design, enabling neural architects to understand hardware compatibility.

## Why IR Matters

The IR decomposes neural networks into ~20 circuit-level primitives, enabling:

- **Accurate energy estimation:** Per-node cost estimation based on OpType
- **Hardware annotation:** Map operations to analog/digital domains
- **D/A boundary detection:** Identify ADC/DAC crossings (energy + quantization cost)
- **Circuit compilation:** Export to Ark for real analog circuit generation

**For hardware architects:** This is your vocabulary for describing what neural computations your primitives can implement.

**For neural architects:** This is how you understand which operations in your model map to analog hardware.

Without IR, we couldn't distinguish between operations that run efficiently in analog vs those that require digital precision.

In [ ]:
import sys
from pathlib import Path

_ROOT = Path.cwd().parent
sys.path.insert(0, str(_ROOT))

import torch
import numpy as np

from neuro_analog.ir import AnalogGraph, make_mvm_node, make_norm_node, make_activation_node
from neuro_analog.ir.types import OpType, Domain, HardwareEstimate
from neuro_analog.ir.energy_model import HardwareProfile, estimate_node_cost

## AnalogGraph Construction

An `AnalogGraph` is a directed graph of `AnalogNode` objects, each representing one operation.

In [ ]:
# Build a simple graph: Linear -> Tanh -> LayerNorm
graph = AnalogGraph(name="simple_mlp")

# Node 1: Matrix-vector multiply (crossbar operation)
mvm = make_mvm_node(
    name="fc1",
    input_shape=(128,),
    output_shape=(256,),
    flops=32768,  # 128 * 256
)
graph.add_node(mvm)

# Node 2: Tanh activation (subthreshold MOSFET differential pair)
act = make_activation_node(
    name="tanh1",
    input_shape=(256,),
    output_shape=(256,),
    op_type=OpType.ANALOG_SIGMOID,
    domain=Domain.ANALOG,
)
graph.add_node(act)

# Node 3: Layer norm (digital-required - precision-critical)
norm = make_norm_node(
    name="ln1",
    input_shape=(256,),
    output_shape=(256,),
    op_type=OpType.LAYER_NORM,
    domain=Domain.DIGITAL,
)
graph.add_node(norm)

print(f"Graph has {len(graph.nodes)} nodes")
for i, node in enumerate(graph.nodes):
    print(f"  Node {i}: {node.op_type.name} in {node.domain.name} domain")

## All OpType Values

The framework defines ~20 primitive operation types spanning all architecture families:

In [ ]:
# Analog-native operations (map directly to circuit primitives)
analog_native = [
    "MVM",                # Matrix-vector multiply → crossbar array
    "INTEGRATION",        # ODE state update → op-amp integrator
    "DECAY",              # Exponential decay → RC circuit
    "ACCUMULATION",       # Signal summation → Kirchhoff current addition
    "ELEMENTWISE_MUL",    # Element-wise multiply → Gilbert cell / analog multiplier
    "ANALOG_SIGMOID",     # Sigmoid → subthreshold MOSFET differential pair
    "ANALOG_EXP",         # Exponential → BJT collector current
    "ANALOG_RELU",        # ReLU → current mirror
    "NOISE_INJECTION",    # Gaussian noise → thermal/shot noise source (TRNG)
    "ANALOG_FIR",         # Short convolution → delay line + summing amplifier
    "SAMPLE",             # Probabilistic sampling → p-bit / sMTJ
    "SKIP_CONNECTION",    # Residual add → current summation
    "GAIN",               # Scalar multiply → programmable gain amplifier
    "GIBBS_STEP",         # Chromatic Gibbs update → subthreshold CMOS RNG
    "RESISTOR_DAC",       # Bias loading → DAC-driven resistor network
]

# Digital-required operations (need digital precision)
digital_required = [
    "SOFTMAX",            # exp + sum + division (precision-critical)
    "LAYER_NORM",         # mean + variance + sqrt + division
    "GROUP_NORM",         # per-group normalization
    "RMS_NORM",           # root mean square normalization
    "BATCH_NORM",         # batch normalization (mean + var + scale + shift)
    "SOFTPLUS",           # log(1 + exp(x))
    "SILU",               # x · σ(x) — SiLU/Swish
    "GELU",               # x · Φ(x)
    "ADALN",              # Adaptive layer norm (scale, shift, gate)
    "DYNAMIC_MATMUL",     # Data-dependent matrix multiply (Q·K^T, attn·V)
    "RESHAPE",            # Tensor reshape/permute (zero-compute, routing)
    "EMBEDDING",          # Token/position embedding lookup
    "MAX_POOL",           # Spatial downsampling
    "DROPOUT",            # Stochastic dropout (training-only, zero-compute at inference)
]

# Hybrid operations (analog-possible with quality loss)
hybrid = [
    "KERNEL_ATTENTION",   # FAVOR+ kernel approximation of softmax attention
    "APPROX_NORM",        # Simplified normalization (L1 norm, fixed stats)
    "PIECEWISE_SILU",     # Piecewise-linear SiLU approximation
    "ANALOG_SOFTMAX",     # Subthreshold exponential + current summing
]

print("Analog-native operations (map to circuits):")
for op in analog_native:
    print(f"  {op}")

print("\nDigital-required operations (need digital precision):")
for op in digital_required:
    print(f"  {op}")

print("\nHybrid operations (analog-possible with quality loss):")
for op in hybrid:
    print(f"  {op}")

print("\n---")
print("For hardware architects: These are the primitives you design and implement.")
print("For neural architects: These are the operations your model will be decomposed into.")

## Domain Annotations

Each node has a `Domain` that determines where it runs:

In [ ]:
domains = [
    (Domain.ANALOG, "Runs in analog circuit (crossbar, integrator, etc.)"),
    (Domain.DIGITAL, "Requires digital computation (precision-critical)"),
    (Domain.HYBRID, "Analog-possible with accuracy tradeoff"),
]

print("Domain annotations and energy implications:")
for domain, description in domains:
    print(f"  {domain.name:10s}: {description}")

## D/A Boundary Detection

D/A boundaries are transitions between analog and digital domains. Each boundary costs:

- **Energy:** ADC/DAC conversion energy (~0.8 pJ per conversion)
- **Quantization error:** Each ADC/DAC crossing adds quantization noise
- **Latency:** ADC/DAC conversion time (~1000 ns per conversion)

In [ ]:
# Count D/A boundaries in our graph
boundaries = 0
for i in range(len(graph.nodes) - 1):
    if graph.nodes[i].domain != graph.nodes[i+1].domain:
        boundaries += 1
        print(f"Boundary at node {i} → {i+1}: {graph.nodes[i].domain.name} → {graph.nodes[i+1].domain.name}")

print(f"\nTotal D/A boundaries: {boundaries}")
print(f"Energy cost per boundary: ~0.8 pJ (ADC) + 0.8 pJ (DAC) = 1.6 pJ")
print(f"Total boundary energy: {boundaries * 1.6} pJ")

## Hardware Noise Annotation

The framework uses mappers to annotate graphs with hardware-specific noise:

In [ ]:
from neuro_analog.mappers import CrossbarMapper, IntegratorMapper, StochasticMapper
from neuro_analog.ir.types import NoiseSpec, CrossbarSpec, IntegratorSpec

# CrossbarMapper: annotates MVM nodes with crossbar noise specifications
crossbar_spec = CrossbarSpec(
    precision_bits=8,
    technology="RRAM",
)
crossbar_mapper = CrossbarMapper(spec=crossbar_spec)

# IntegratorMapper: annotates INTEGRATION nodes with thermal noise (kT/C)
integrator_spec = IntegratorSpec(
    time_constant_s=1e-3,
    bandwidth_hz=250e3,
)
integrator_mapper = IntegratorMapper(spec=integrator_spec)

# StochasticMapper: annotates SAMPLE/NOISE_INJECTION nodes with shot noise
stochastic_mapper = StochasticMapper(
    i_bias_ua=1.0,  # Bias current in microamps
    bandwidth_hz=1e6,  # Bandwidth in Hz
)

print("Hardware noise mappers:")
print(f"  CrossbarMapper: precision_bits={crossbar_mapper.spec.precision_bits}, technology={crossbar_mapper.spec.technology}")
print(f"  IntegratorMapper: time_constant={integrator_mapper.spec.time_constant_s*1e3:.1f} ms, bandwidth={integrator_mapper.spec.bandwidth_hz/1e3:.0f} kHz")
print(f"  StochasticMapper: i_bias={stochastic_mapper.i_bias_ua} µA, bandwidth={stochastic_mapper.bandwidth_hz/1e6:.1f} MHz")

## NoiseSpec Parameters

`NoiseSpec` defines the physical noise characteristics for an operation:

In [ ]:
# Example NoiseSpec for thermal noise
thermal_spec = NoiseSpec(
    kind="thermal",
    sigma=0.02,  # Noise std-dev in equivalent input units
    mean=0.0,  # Bias / systematic offset
    corr_length=1e-6,  # Temporal correlation length (1 µs)
    corr_spatial=0.1,  # Spatial correlation across rows/cols
    bandwidth_hz=250e3,  # Effective noise bandwidth
)

print("NoiseSpec parameters and physical meaning:")
print(f"  kind: {thermal_spec.kind} - Noise source type")
print(f"  sigma: {thermal_spec.sigma} - Noise std-dev (equivalent input units)")
print(f"  mean: {thermal_spec.mean} - Systematic offset/bias")
print(f"  corr_length: {thermal_spec.corr_length*1e6:.1f} µs - Temporal correlation")
print(f"  corr_spatial: {thermal_spec.corr_spatial} - Spatial correlation")
print(f"  bandwidth: {thermal_spec.bandwidth_hz/1e3:.0f} kHz - Effective bandwidth")

## Per-Node Cost Estimation

Using `estimate_node_cost()`, we can compute energy and latency for each node:

In [ ]:
# Load hardware profile
profile = HardwareProfile()

# Estimate cost for each node
print("Per-node energy and latency estimates:")
print(f"{'Node':>10} {'OpType':>15} {'Domain':>10} {'Energy (pJ)':>12} {'Latency (ns)':>12}")
print("-" * 65)

total_energy = 0
total_latency = 0

for node in graph.nodes:
    estimate = estimate_node_cost(node, profile)
    total_energy += estimate.energy_pJ
    total_latency += estimate.latency_ns
    print(f"{node.name:10s} {node.op_type.name:15s} {node.domain.name:10s} {estimate.energy_pJ:12.0f} {estimate.latency_ns:12.0f}")

print("-" * 65)
print(f"{'TOTAL':>10} {'':15s} {'':10s} {total_energy:12.0f} {total_latency:12.0f}")

## Graph Analysis Methods

The `AnalogGraph` provides analysis methods to understand the graph structure:

In [ ]:
# Analyze flop fractions by domain
flop_analysis = graph.flop_fractions()

print("FLOP breakdown by domain:")
print(f"  Analog FLOPs: {flop_analysis['analog_flops']:,} ({flop_analysis['analog_fraction']*100:.1f}%)")
print(f"  Digital FLOPs: {flop_analysis['digital_flops']:,} ({flop_analysis['digital_fraction']*100:.1f}%)")
print(f"  Hybrid FLOPs: {flop_analysis['hybrid_flops']:,} ({flop_analysis['hybrid_fraction']*100:.1f}%)")

# Analyze with hardware profile
analysis = graph.analyze(profile)

print("\nHardware analysis:")
print(f"  Total analog energy: {analysis['total_analog_energy_pJ']:,.0f} pJ")
print(f"  Total digital energy: {analysis['total_digital_energy_pJ']:,.0f} pJ")
print(f"  Energy saving: {analysis['energy_saving_vs_digital']*100:.1f}%")

## Energy/Latency Breakdown: Analog vs Digital

The key energy advantage comes from analog operations:

In [ ]:
print("Energy comparison (per operation):")
print(f"  Analog MAC: {profile.analog_mac_energy_pJ} pJ (IBM PCM arrays)")
print(f"  Digital MAC: {profile.digital_mac_energy_pJ} pJ (GPU/SRAM-IMC baselines)")
print(f"  Speedup: {profile.digital_mac_energy_pJ / profile.analog_mac_energy_pJ:.1f}x")

print("\nLatency comparison (per operation):")
analog_throughput = profile.analog_mac_throughput  # MAC/s
digital_throughput = profile.digital_mac_throughput  # MAC/s
print(f"  Analog throughput: {analog_throughput/1e12:.1f} TOPS")
print(f"  Digital throughput: {digital_throughput/1e11:.1f} GOPS")
print(f"  Speedup: {analog_throughput/digital_throughput:.1f}x")

## Building a More Complex Graph

Let's build a graph with more nodes to see the analysis in action:

In [ ]:
# Build a more complex graph (simulating a small neural network)
complex_graph = AnalogGraph(name="complex_example")

# Layer 1: MVM -> Tanh
complex_graph.add_node(make_mvm_node("fc1", (784,), (512,), flops=401408))
complex_graph.add_node(make_activation_node("tanh1", (512,), (512,), OpType.ANALOG_SIGMOID, Domain.ANALOG))

# Layer 2: MVM -> Tanh
complex_graph.add_node(make_mvm_node("fc2", (512,), (256,), flops=131072))
complex_graph.add_node(make_activation_node("tanh2", (256,), (256,), OpType.ANALOG_SIGMOID, Domain.ANALOG))

# Layer 3: MVM -> LayerNorm (D/A boundary)
complex_graph.add_node(make_mvm_node("fc3", (256,), (128,), flops=32768))
complex_graph.add_node(make_norm_node("ln1", (128,), (128,), OpType.LAYER_NORM, Domain.DIGITAL))

# Layer 4: MVM -> Softmax (digital-required)
complex_graph.add_node(make_mvm_node("fc4", (128,), (10,), flops=1280))
complex_graph.add_node(make_activation_node("softmax", (10,), (10,), OpType.SOFTMAX, Domain.DIGITAL))

print(f"Complex graph has {len(complex_graph.nodes)} nodes")

# Analyze
complex_analysis = complex_graph.analyze(profile)
print(f"\nAnalog FLOP fraction: {complex_graph.flop_fractions()['analog_fraction']*100:.1f}%")
print(f"Energy saving: {complex_analysis['energy_saving_vs_digital']*100:.1f}%")

## Key Takeaways

1. **IR decomposes models into ~20 circuit-level primitives** for accurate energy estimation
2. **OpType classification determines analog vs digital mapping** (analog-native, digital-required, hybrid)
3. **Domain annotations identify where operations run** (ANALOG, DIGITAL, HYBRID)
4. **D/A boundaries are expensive** - each ADC/DAC crossing costs energy and adds quantization error
5. **Hardware mappers add physical noise** (CrossbarMapper, IntegratorMapper, StochasticMapper)
6. **Per-node cost estimation enables accurate energy/latency modeling**
7. **Analog MAC is 20x more efficient than digital** (5 pJ vs 100 pJ)
8. **Graph analysis methods provide high-level metrics** (flop fractions, energy savings)